In [0]:
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
combined_path = "/Workspace/Users/jezapataf@eafit.edu.co/ProyectoIntegrador2/notebooks/data/processed/financial_sentiment_combined.parquet"

path = Path(combined_path)

if not path.exists():
    raise FileNotFoundError(f"No se encontró el archivo en: {combined_path}")

print("Archivo encontrado:", combined_path)

In [0]:
pdf = pd.read_parquet(combined_path)

print("Filas:", len(pdf))
print("Columnas:", list(pdf.columns))

#display(pdf.head(10))

In [0]:
catalog = "workspace"
schema = "financial_sentiment"

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    table_prefix = f"{catalog}.{schema}"
except Exception as e:
    print("No se pudo usar catálogo workspace. Se usará schema simple.")
    print(e)
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema}")
    table_prefix = schema

print("Tablas se crearán en:", table_prefix)

In [0]:
bronze_df = spark.createDataFrame(pdf)

bronze_df = bronze_df.select(
    F.col("dataset_id").cast("string").alias("dataset_id"),
    F.col("dataset_label").cast("string").alias("dataset_label"),
    F.col("source_platform").cast("string").alias("source_platform"),
    F.col("split").cast("string").alias("split"),
    F.col("text").cast("string").alias("text"),
    F.col("label_normalized").cast("string").alias("label_normalized")
).withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

bronze_df.write.mode("overwrite").saveAsTable(
    f"{table_prefix}.bronze_financial_sentiment_raw"
)

print("Tabla bronze creada")
display(spark.table(f"{table_prefix}.bronze_financial_sentiment_raw").limit(10))

In [0]:
silver_df = bronze_df.select(
    "dataset_id",
    "dataset_label",
    "source_platform",
    F.lower(F.trim(F.col("split"))).alias("split"),
    F.trim(F.col("text")).alias("text_raw"),
    F.regexp_replace(F.lower(F.trim(F.col("text"))), r"\s+", " ").alias("text_clean"),
    F.lower(F.trim(F.col("label_normalized"))).alias("label_normalized"),
    "ingestion_timestamp"
).withColumn(
    "text_hash",
    F.sha2(F.col("text_clean"), 256)
).filter(
    F.col("text_clean").isNotNull()
).filter(
    F.length(F.col("text_clean")) > 0
).filter(
    F.col("label_normalized").isin("negative", "neutral", "positive")
).filter(
    F.col("split").isin("train", "validation", "test")
).dropDuplicates(
    ["text_hash", "label_normalized", "split"]
)

silver_df.write.mode("overwrite").saveAsTable(
    f"{table_prefix}.silver_financial_sentiment_canonical"
)

print("Tabla silver creada")
display(spark.table(f"{table_prefix}.silver_financial_sentiment_canonical").limit(10))

In [0]:
df = spark.table(f"{table_prefix}.silver_financial_sentiment_canonical")

total = df.count()
null_texts = df.filter(F.col("text_clean").isNull() | (F.length(F.col("text_clean")) == 0)).count()
invalid_labels = df.filter(~F.col("label_normalized").isin("negative", "neutral", "positive")).count()
invalid_splits = df.filter(~F.col("split").isin("train", "validation", "test")).count()

label_conflicts = df.groupBy("text_hash").agg(
    F.countDistinct("label_normalized").alias("distinct_labels")
).filter(
    F.col("distinct_labels") > 1
).count()

cross_split_duplicates = df.groupBy("text_hash").agg(
    F.countDistinct("split").alias("distinct_splits")
).filter(
    F.col("distinct_splits") > 1
).count()

print("Total registros silver:", total)
print("Textos nulos o vacíos:", null_texts)
print("Labels inválidos:", invalid_labels)
print("Splits inválidos:", invalid_splits)
print("Textos con conflicto de label:", label_conflicts)
print("Textos repetidos en varios splits:", cross_split_duplicates)

if total == 0:
    raise Exception("La tabla silver quedó vacía.")

if null_texts > 0:
    raise Exception("Hay textos nulos o vacíos.")

if invalid_labels > 0:
    raise Exception("Hay labels inválidos.")

if invalid_splits > 0:
    raise Exception("Hay splits inválidos.")



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table(f"{table_prefix}.silver_financial_sentiment_canonical")

# 1. Identificar textos con conflicto de etiqueta
label_conflict_hashes = (
    silver_df
    .groupBy("text_hash")
    .agg(
        F.countDistinct("label_normalized").alias("distinct_labels"),
        F.collect_set("label_normalized").alias("labels_detected"),
        F.count("*").alias("rows_count")
    )
    .filter(F.col("distinct_labels") > 1)
)

label_conflict_details = (
    silver_df
    .join(label_conflict_hashes.select("text_hash"), on="text_hash", how="inner")
    .select(
        "text_hash",
        "text_raw",
        "text_clean",
        "label_normalized",
        "split",
        "dataset_id",
        "dataset_label",
        "source_platform"
    )
)

label_conflict_details.write.mode("overwrite").saveAsTable(
    f"{table_prefix}.dq_label_conflicts"
)

print("Tabla de conflictos de label creada:")
print(f"{table_prefix}.dq_label_conflicts")

display(label_conflict_details.orderBy("text_hash", "label_normalized", "split").limit(50))

In [0]:
cross_split_hashes = (
    silver_df
    .groupBy("text_hash")
    .agg(
        F.countDistinct("split").alias("distinct_splits"),
        F.collect_set("split").alias("splits_detected"),
        F.count("*").alias("rows_count")
    )
    .filter(F.col("distinct_splits") > 1)
)

cross_split_details = (
    silver_df
    .join(cross_split_hashes.select("text_hash"), on="text_hash", how="inner")
    .select(
        "text_hash",
        "text_raw",
        "text_clean",
        "label_normalized",
        "split",
        "dataset_id",
        "dataset_label",
        "source_platform"
    )
)

cross_split_details.write.mode("overwrite").saveAsTable(
    f"{table_prefix}.dq_cross_split_duplicates"
)

print("Tabla de duplicados entre splits creada:")
print(f"{table_prefix}.dq_cross_split_duplicates")

display(cross_split_details.orderBy("text_hash", "split").limit(50))

In [0]:
silver_without_label_conflicts = (
    silver_df
    .join(
        label_conflict_hashes.select("text_hash"),
        on="text_hash",
        how="left_anti"
    )
)

print("Registros silver originales:", silver_df.count())
print("Registros sin conflictos de label:", silver_without_label_conflicts.count())

In [0]:
silver_ranked = silver_without_label_conflicts.withColumn(
    "split_priority",
    F.when(F.col("split") == "test", F.lit(1))
     .when(F.col("split") == "validation", F.lit(2))
     .when(F.col("split") == "train", F.lit(3))
     .otherwise(F.lit(4))
)

window_spec = Window.partitionBy("text_hash").orderBy(
    "split_priority",
    "dataset_id",
    "label_normalized"
)

gold_df = (
    silver_ranked
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "split_priority")
)

gold_df.write.mode("overwrite").saveAsTable(
    f"{table_prefix}.gold_financial_sentiment_training"
)

print("Tabla gold creada:")
print(f"{table_prefix}.gold_financial_sentiment_training")

display(spark.table(f"{table_prefix}.gold_financial_sentiment_training").limit(10))

In [0]:
gold = spark.table(f"{table_prefix}.gold_financial_sentiment_training")

gold_total = gold.count()

gold_label_conflicts = (
    gold
    .groupBy("text_hash")
    .agg(F.countDistinct("label_normalized").alias("distinct_labels"))
    .filter(F.col("distinct_labels") > 1)
    .count()
)

gold_cross_split_duplicates = (
    gold
    .groupBy("text_hash")
    .agg(F.countDistinct("split").alias("distinct_splits"))
    .filter(F.col("distinct_splits") > 1)
    .count()
)

print("Total gold:", gold_total)
print("Conflictos de label en gold:", gold_label_conflicts)
print("Duplicados entre splits en gold:", gold_cross_split_duplicates)

if gold_total == 0:
    raise Exception("La tabla gold quedó vacía.")

if gold_label_conflicts > 0:
    raise Exception("Gold todavía tiene conflictos de label.")

if gold_cross_split_duplicates > 0:
    raise Exception("Gold todavía tiene textos repetidos entre splits.")

In [0]:
display(
    gold.groupBy("split")
        .count()
        .orderBy("split")
)

display(
    gold.groupBy("label_normalized")
        .count()
        .orderBy("label_normalized")
)

display(
    gold.groupBy("split", "label_normalized")
        .count()
        .orderBy("split", "label_normalized")
)